# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder)without this

/Users/mac/Documents/dev/ID2221/dic/Week 3


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 12:35:53 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/22 12:35:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9f542c53-b836-4b5f-b62f-5d605d3c0990;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips_part', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Load dataset file

In [5]:
# load data/weather.csv into a DataFrame
weather_df = spark.read.csv("../data/weather.csv", header=True, inferSchema=True)
weather_df.show(5)
weather_df.printSchema()

+----+-----+---+----+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+------+-----------+----+-----------+----+-----------+
|year|month|day|hour|temp|temp_source|rhum|rhum_source|prcp|prcp_source|snwd|snwd_source|wdir|wdir_source|wspd|wspd_source|wpgt|wpgt_source|  pres|pres_source|cldc|cldc_source|coco|coco_source|
+----+-----+---+----+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+------+-----------+----+-----------+----+-----------+
|2024|    1|  1|   0| 6.1|   isd_lite|  47|   isd_lite| 0.0| dwd_mosmix|NULL|       NULL| 280|   isd_lite| 9.4|   isd_lite|NULL|       NULL|1015.8|   isd_lite|   8|   isd_lite|   3| dwd_mosmix|
|2024|    1|  1|   1| 6.1|   isd_lite|  45|   isd_lite| 0.0|   isd_lite|NULL|       NULL| 260|   isd_lite|13.0|   isd_lite|NULL|       NULL|1015.8|   isd_lite|   7| dwd_mosmix|   3| dwd_mosmix|
|2024|    1|  1|   2| 6.1|   i

# Summary of dataset

Note that negative values do occur in original dataset so synthetic also having some is to be expected

In [6]:
# summary of dataset
weather_df.describe().show()

26/09/22 12:38:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------+------------------+------------------+------------------+------------------+-----------+-----------------+-----------+------------------+-----------+----+-----------+------------------+-----------+-----------------+-----------+----+-----------+------------------+-----------+-----------------+-----------+-----------------+-----------+
|summary|  year|             month|               day|              hour|              temp|temp_source|             rhum|rhum_source|              prcp|prcp_source|snwd|snwd_source|              wdir|wdir_source|             wspd|wspd_source|wpgt|wpgt_source|              pres|pres_source|             cldc|cldc_source|             coco|coco_source|
+-------+------+------------------+------------------+------------------+------------------+-----------+-----------------+-----------+------------------+-----------+----+-----------+------------------+-----------+-----------------+-----------+----+-----------+------------------+-----------+-----

# Generate rows

Decimal values are handled with column values from normal distribution (mean, std) delimited by observed real range  

String values are handled with categorical distribution

Integer values are handled with categorical distribution

In [10]:
# get latest date in the dataset based on columns year, month (1-12), day (1-31)
latest_date = (
    weather_df
    .withColumn(
        "date",
        F.make_date(
            F.col("year"),
            F.col("month"),
            F.col("day")
        )
    )
    .agg(F.max("date").alias("latest_date"))
    .first()["latest_date"]
)

print(latest_date)

2024-12-31


In [24]:
import random
import string
from datetime import timedelta
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, TimestampType, TimestampNTZType, FloatType, ShortType, DecimalType

number_of_rows = int(1.0 * weather_df.count())


existing_values_and_stats = {}



# find the existing values and stats for each column in weather_df
for field in weather_df.schema.fields:
    if isinstance(field.dataType, StringType):
        existing_values_and_stats[field.name] = {
            "Values": weather_df.select(field.name).distinct().rdd.flatMap(lambda x: x).collect()
        }
    elif isinstance(field.dataType, (IntegerType, LongType, ShortType)):
        existing_values = weather_df.select(field.name).distinct().rdd.flatMap(lambda x: x).collect()
        existing_values_and_stats[field.name] = {
            "Values": existing_values
        }
    elif isinstance(field.dataType, (DoubleType, FloatType, DecimalType)):
        # calculate mean and std of the column
        mean_std = weather_df.select(F.mean(field.name), F.stddev(field.name)).first()
        mean = mean_std[0] if mean_std else None
        stddev = mean_std[1] if mean_std else None
        if mean is None or stddev is None:
            # error
            raise ValueError(f"Cannot calculate mean and stddev for column {field.name}")
        else:
            existing_values_and_stats[field.name] = {
                "Mean": mean,
                "Stddev": stddev,
                "max": weather_df.select(F.max(field.name)).first()[0],
                "min": weather_df.select(F.min(field.name)).first()[0]
            }
    else:
        # error for unsupported data types
        raise ValueError(f"Unsupported data type: {field.dataType}")
    
print("Computed existing values and stats for each column in weather_df")
    

from pyspark.sql import functions as F


def uniform_from_values(values, data_type, seed=None):
    n = len(values)

    return F.element_at(
        F.array(*[F.lit(v).cast(data_type) for v in values]),
        (F.rand(seed) * n).cast("int") + 1,
    )

def normal_from_stats(existing_values_and_stats, column_name, seed=None):
    mean = existing_values_and_stats[column_name]["Mean"]
    stddev = existing_values_and_stats[column_name]["Stddev"]
    min_value = existing_values_and_stats[column_name]["min"]
    max_value = existing_values_and_stats[column_name]["max"]

    return F.least(
        F.lit(max_value),
        F.greatest(
            F.lit(min_value),
            F.lit(mean) + F.lit(stddev) * F.randn(seed)
        )
    )


seed = 42
generated_df = (
    spark.range(number_of_rows)
    # Timestamp columns: generate timestamps based on the latest date in the dataset
    .withColumn(
        "ts",
        F.expr(f"timestamp('{latest_date}') + id * interval 1 hour")
    )
    .withColumn("year", F.year("ts").cast("int"))
    .withColumn("month", F.month("ts").cast("int"))
    .withColumn("day", F.dayofmonth("ts").cast("int"))
    .withColumn("hour", F.hour("ts").cast("int"))
    # Categorical columns: pick random value from the existing values in the column
    .withColumn(
        "temp_source",
        uniform_from_values(existing_values_and_stats["temp_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "rhum",
        uniform_from_values(existing_values_and_stats["rhum"]["Values"], IntegerType(), seed)
    )
    .withColumn(
        "rhum_source",
        uniform_from_values(existing_values_and_stats["rhum_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "prcp_source",
        uniform_from_values(existing_values_and_stats["prcp_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "snwd_source",
        uniform_from_values(existing_values_and_stats["snwd_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "wdir",
        uniform_from_values(existing_values_and_stats["wdir"]["Values"], IntegerType(), seed)
    )
    .withColumn(
        "wdir_source",
        uniform_from_values(existing_values_and_stats["wdir_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "wspd_source",
        uniform_from_values(existing_values_and_stats["wspd_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "wpgt",
        uniform_from_values(existing_values_and_stats["wpgt"]["Values"], StringType(), seed)
    )
    .withColumn(
        "wpgt_source",
        uniform_from_values(existing_values_and_stats["wpgt_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "pres_source",
        uniform_from_values(existing_values_and_stats["pres_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "cldc",
        uniform_from_values(existing_values_and_stats["cldc"]["Values"], IntegerType(), seed)
    )
    .withColumn(
        "cldc_source",
        uniform_from_values(existing_values_and_stats["cldc_source"]["Values"], StringType(), seed)
    )
    .withColumn(
        "coco",
        uniform_from_values(existing_values_and_stats["coco"]["Values"], IntegerType(), seed)
    )
    .withColumn(
        "coco_source",
        uniform_from_values(existing_values_and_stats["coco_source"]["Values"], StringType(), seed)
    )
    # Numerical columns: generate random value from normal distribution with mean and stddev delimited by the observed min and max values of the column
    .withColumn(
        "temp",
        normal_from_stats(existing_values_and_stats, "temp", seed)
    )
    .withColumn(
        "prcp",
        normal_from_stats(existing_values_and_stats, "prcp", seed)
    )
    .withColumn(
        "wspd",
        normal_from_stats(existing_values_and_stats, "wspd", seed)
    )
    .withColumn(
        "pres",
        normal_from_stats(existing_values_and_stats, "pres", seed)
    )
    .withColumn( # special case, all zero or null values?
        "snwd",
        # set to zero always
        F.lit(0.0).cast(DoubleType())
    )
    # new column
    .withColumn( 
        "humidity",
        # set to value of rhum column, but between 20 and 100
        F.when(F.col("rhum") < 20, 20)
        .when(F.col("rhum") > 100, 100)
        .otherwise(F.col("rhum"))
    )
)

# drop the id column generated by spark.range
generated_df = generated_df.drop("id")
generated_df = generated_df.drop("ts")

# reorder the columns to match the original weather_df, also make sure to get the new columns
generated_df = generated_df.select(weather_df.columns + ["humidity"])



generated_df.show(30)
generated_df.printSchema()

Computed existing values and stats for each column in weather_df
+----+-----+---+----+------------------+-----------+----+-----------+--------------------+-----------+----+-----------+----+-----------+------------------+-----------+----+-----------+------------------+-----------+----+-----------+----+-----------+--------+
|year|month|day|hour|              temp|temp_source|rhum|rhum_source|                prcp|prcp_source|snwd|snwd_source|wdir|wdir_source|              wspd|wspd_source|wpgt|wpgt_source|              pres|pres_source|cldc|cldc_source|coco|coco_source|humidity|
+----+-----+---+----+------------------+-----------+----+-----------+--------------------+-----------+----+-----------+----+-----------+------------------+-----------+----+-----------+------------------+-----------+----+-----------+----+-----------+--------+
|2024|   12| 31|   0|              35.6|   isd_lite|  45|   isd_lite|   2.277189227202875|   isd_lite| 0.0|       NULL|  25| dwd_mosmix|36.985502736781115| dw

# Write dataframe to csv file
Note: crc files are only metadata

In [ ]:
# save generated_df to single csv file
generated_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("../updates/weather_update")